# YogaPose StyleGAN2-ADA Colab (A100-ready)

This notebook trains a **StyleGAN2-ADA** model on a **subset of 5–10 yoga pose classes** at **256×256** when feasible.

## Recommended runtime
- **GPU:** NVIDIA A100 preferred
- TPU is **not recommended** for this workflow

## What this notebook does
1. Mounts Google Drive
2. Downloads / reuses the `arrowe/yoga-poses-dataset-107` zip on Drive
3. Extracts dataset and selects 5–10 yoga pose classes
4. Builds a StyleGAN2-ADA training dataset ZIP
5. Clones StyleGAN2-ADA
6. Launches training with Drive-backed outputs
7. Generates sample images after training

> Note: a short run under 30 minutes will produce a **prototype** model, not a fully converged generator.

---
## Cell 1 — Mount Drive and configure paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive/YogaPoseStyleGAN'
DATA_ZIP = '/content/drive/MyDrive/computer-vision/yoga-poses-dataset-107.zip'
EXTRACT_DIR = os.path.join(DRIVE_ROOT, 'dataset')
SUBSET_DIR = os.path.join(DRIVE_ROOT, 'subset_dataset')
STYLEGAN_REPO = '/content/stylegan2-ada-pytorch'
TRAIN_ZIP = os.path.join(DRIVE_ROOT, 'yoga_subset_stylegan.zip')
RUNS_DIR = os.path.join(DRIVE_ROOT, 'training-runs')
GENERATED_DIR = os.path.join(DRIVE_ROOT, 'generated_samples')

for d in [DRIVE_ROOT, EXTRACT_DIR, SUBSET_DIR, RUNS_DIR, GENERATED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted ✓')
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('DATA_ZIP     =', DATA_ZIP)
print('EXTRACT_DIR  =', EXTRACT_DIR)
print('SUBSET_DIR   =', SUBSET_DIR)
print('TRAIN_ZIP    =', TRAIN_ZIP)
print('RUNS_DIR     =', RUNS_DIR)

---
## Cell 2 — Install dependencies

In [ ]:
!pip install -q pillow tqdm

---
## Cell 3 — Download dataset zip if needed

Uses the Kaggle endpoint you provided.

In [ ]:
if not os.path.exists(DATA_ZIP):
    print('Downloading dataset zip...')
    !curl -L https://www.kaggle.com/api/v1/datasets/download/arrowe/yoga-poses-dataset-107 -o "{DATA_ZIP}"
else:
    print('Dataset zip already exists ✓')

print('Zip path:', DATA_ZIP)

---
## Cell 4 — Extract dataset and discover train directory

In [ ]:
import glob, zipfile

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
if not train_dirs:
    print('Extracting dataset...')
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Dataset already extracted ✓')

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
DATASET_DIR = train_dirs[0] if train_dirs else EXTRACT_DIR
print('DATASET_DIR =', DATASET_DIR)

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print('Total classes found:', len(classes))
print(classes[:20])

---
## Cell 5 — Choose 5–10 yoga poses and image size

For a fast A100 run, use 5–10 classes. 256×256 is included as recommended.

In [ ]:
# ⚙️ Edit these
SELECTED_CLASSES = [
    'adho mukha svanasana',
    'vrksasana',
    'trikonasana',
    'virabhadrasana ii',
    'balasana',
    'bhujangasana',
    'dhanurasana',
    'bakasana'
]

IMG_SIZE = 256   # recommended for A100
MAX_IMAGES_PER_CLASS = 300   # keeps run practical

print('Selected classes:', SELECTED_CLASSES)
print('IMG_SIZE:', IMG_SIZE)
print('MAX_IMAGES_PER_CLASS:', MAX_IMAGES_PER_CLASS)

---
## Cell 6 — Build subset dataset (copy + resize to square)

In [ ]:
import shutil
from PIL import Image, ImageOps
from tqdm import tqdm

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Clear previous subset so the run is reproducible
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
os.makedirs(SUBSET_DIR, exist_ok=True)

def resize_and_pad(img, size):
    img = ImageOps.exif_transpose(img).convert('RGB')
    img.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

summary = {}
for cls in SELECTED_CLASSES:
    src_dir = os.path.join(DATASET_DIR, cls)
    dst_dir = os.path.join(SUBSET_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f'WARNING: class not found: {cls}')
        continue

    files = [f for f in sorted(os.listdir(src_dir)) if os.path.splitext(f)[1].lower() in VALID_EXTS]
    files = files[:MAX_IMAGES_PER_CLASS]

    count = 0
    for fname in tqdm(files, desc=cls):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, os.path.splitext(fname)[0] + '.png')
        try:
            img = Image.open(src)
            out = resize_and_pad(img, IMG_SIZE)
            out.save(dst, format='PNG')
            count += 1
        except Exception:
            pass

    summary[cls] = count

print('Subset build complete ✓')
print(summary)

---
## Cell 7 — Create StyleGAN training ZIP

StyleGAN2-ADA expects a dataset zip or folder. We'll create a zip on Drive.

In [ ]:
import zipfile

if os.path.exists(TRAIN_ZIP):
    os.remove(TRAIN_ZIP)

with zipfile.ZipFile(TRAIN_ZIP, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, _, files in os.walk(SUBSET_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, SUBSET_DIR)
            zf.write(full, rel)

print('Training zip created ✓')
print(TRAIN_ZIP)

---
## Cell 8 — Clone StyleGAN2-ADA

In [ ]:
if not os.path.exists(STYLEGAN_REPO):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git {STYLEGAN_REPO}
else:
    print('StyleGAN2-ADA repo already cloned ✓')

%cd {STYLEGAN_REPO}

---
## Cell 9 — Check GPU and choose training settings

In [ ]:
!nvidia-smi

# Fast prototype settings for <30 min A100 run
KIMG = 200
SNAP = 20
BATCH = 16
AUG = 'ada'

print('Training settings:')
print('KIMG =', KIMG)
print('SNAP =', SNAP)
print('BATCH=', BATCH)
print('AUG  =', AUG)

---
## Cell 10 — Launch training

This starts a short prototype run. Outputs and checkpoints are saved to Drive.

In [ ]:
!python train.py \
  --outdir={RUNS_DIR} \
  --data={TRAIN_ZIP} \
  --gpus=1 \
  --cfg=auto \
  --snap={SNAP} \
  --kimg={KIMG} \
  --batch={BATCH} \
  --aug={AUG}

---
## Cell 11 — Locate latest network snapshot

In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(RUNS_DIR, '*')))
latest_run = run_dirs[-1] if run_dirs else None
print('Latest run:', latest_run)

snapshots = sorted(glob.glob(os.path.join(latest_run, 'network-snapshot-*.pkl'))) if latest_run else []
NETWORK_PKL = snapshots[-1] if snapshots else None
print('Latest snapshot:', NETWORK_PKL)

---
## Cell 12 — Generate sample images

In [ ]:
if NETWORK_PKL is None:
    print('No snapshot found. Train first.')
else:
    !python generate.py --outdir={GENERATED_DIR} --trunc=0.8 --seeds=0-24 --network={NETWORK_PKL}

---
## Cell 13 — Show generated samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_files = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))[:16]

if not sample_files:
    print('No generated samples found.')
else:
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, f in zip(axes.flatten(), sample_files):
        ax.imshow(mpimg.imread(f))
        ax.set_title(os.path.basename(f), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()